# 01. Variant distribution across genomic windows

Summarize retained variants in genomic windows and inspect chromosome profiles. The input table contains variant coordinates and minor allele frequencies (MAF); outputs include window statistics, figures, and the tables used for plotting.

# 01. 基因组窗口中的变异分布

按基因组窗口汇总保留的变异，查看染色体上的分布。输入表包含变异坐标及次要等位基因频率（MAF）；输出包括窗口统计、图件及实际用于绘图的数据表。

## 1. Variant data

### 1.1 Analysis setup

## 1. 变异数据

### 1.1 分析准备

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import socket
import sys
from datetime import datetime
from time import perf_counter
import pandas as pd

%load_ext autoreload
%autoreload 2

project_dir = '/absolute/remote/project/root'  # Replace with the confirmed config path.
os.chdir(project_dir)
project_dir = os.getcwd()
data_dir = f'{project_dir}/data'
source_dir = f'{data_dir}/source'
results_dir = f'{data_dir}/results'

os.makedirs(data_dir, exist_ok=True)
os.makedirs(source_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from src.plot_style import set_plot_style
set_plot_style()

print(f'Host: {socket.gethostname()}')
print(f'Working directory: {os.getcwd()}')
print(f'Python: {sys.executable}')
print(f"Environment prefix: {os.environ.get('CONDA_PREFIX', '<not set>')}")

### 1.2 Variant coordinates and allele frequencies

Each row represents a unique variant with its chromosome, position, and minor allele frequency (MAF). Positions use 1-based coordinates; MAF ranges from 0 to 0.5, with missing values allowed.

### 1.2 变异坐标与等位基因频率

每行对应一个唯一变异，包含染色体、位置及次要等位基因频率（MAF）。位置采用 1-based 坐标；MAF 范围为 0 到 0.5，可存在缺失值。

In [ ]:
variant_file = f'{source_dir}/variants.csv'
variant_table_h5 = f'{results_dir}/variant_table.h5'
start_time = perf_counter()

if os.path.exists(variant_table_h5):
    variants_df = pd.read_hdf(variant_table_h5, key='df')
    print(f'[{datetime.now():%H:%M:%S}] Reusing: {variant_table_h5}')
else:
    variants_df = pd.read_csv(variant_file, dtype={'CHROM': str}, usecols=['CHROM', 'POS', 'MAF'])
    temporary_file = f'{variant_table_h5}.tmp'
    variants_df.to_hdf(temporary_file, key='df', mode='w')
    os.replace(temporary_file, variant_table_h5)
    print(f'[{datetime.now():%H:%M:%S}] Saved: {variant_table_h5}')

print(f'Variants: {len(variants_df):,}; elapsed: {perf_counter() - start_time:.1f}s')
variants_df.head()

## 2. Window-level variant summaries

Summarize variants in contiguous 500 bp windows (1–500, 501–1000, …) to describe their distribution along chromosomes. Retain variants with MAF ≥ 0.05 and exclude missing MAF. Each window reports the number of retained variants and their mean MAF; windows without retained variants are omitted.

## 2. 窗口水平的变异汇总

采用连续的 500 bp 窗口（1–500、501–1000、……）汇总变异，描述其沿染色体的分布。保留 MAF ≥ 0.05 的变异，排除 MAF 缺失值。每个窗口报告保留变异的数量及平均 MAF；没有保留变异的窗口不纳入汇总。

### 2.1 Window scale and allele-frequency threshold

### 2.1 窗口尺度与等位基因频率阈值

In [ ]:
window_size = 500
min_maf = 0.05
summary_file = f'{results_dir}/variant_windows.b{window_size}.maf{min_maf}.csv.gz'

### 2.2 Summarize variants by chromosome and window

### 2.2 按染色体及窗口汇总变异

In [ ]:
from src.style_examples import build_chromosome_summaries

if os.path.exists(summary_file):
    summary_df = pd.read_csv(summary_file, dtype={'CHROM': str})
    print(f'[{datetime.now():%H:%M:%S}] Reusing: {summary_file}')
else:
    summary_df = build_chromosome_summaries(
        variants_df, window_size=window_size, min_maf=min_maf,
    )
    temporary_file = f'{summary_file}.tmp'
    summary_df.to_csv(temporary_file, index=False, compression='gzip')
    os.replace(temporary_file, summary_file)
    print(f'[{datetime.now():%H:%M:%S}] Saved: {summary_file}')

print(f'Occupied windows: {len(summary_df):,}; columns: {summary_df.columns.tolist()}')
summary_df.head()

## 3. Chromosome profiles

Compare the mean MAF of retained variants across windows on selected chromosomes. The x-axis gives the window start in megabases (Mb); the y-axis gives mean MAF.

## 3. 染色体上的分布

比较选定染色体上不同窗口中保留变异的平均 MAF。横坐标表示窗口起点，单位为 Mb；纵坐标表示平均 MAF。

In [ ]:
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from src.style_examples import plot_window_maf

summary_df = pd.read_csv(summary_file, dtype={'CHROM': str})

chromosomes = ['Chr1', 'Chr2']  # Match identifiers in the input table.
for chromosome in tqdm(chromosomes, desc='Exporting figures', unit='figure'):
    figure_prefix = f'{results_dir}/variant_windows.b{window_size}.maf{min_maf}.{chromosome}'
    fig, ax = plt.subplots(figsize=(8, 3))
    plot_window_maf(summary_df, chromosome, ax)
    ax.set_ylim(0, 0.5)
    fig.tight_layout()

    plot_df = summary_df.loc[summary_df['CHROM'] == chromosome].sort_values('WINDOW_START').copy()
    plot_df['POSITION_MB'] = plot_df['WINDOW_START'] / 1e6
    plot_df.to_csv(f'{figure_prefix}.csv.gz', index=False, compression='gzip')
    fig.savefig(f'{figure_prefix}.pdf')
    fig.savefig(f'{figure_prefix}.png', dpi=200)
    plt.show()
    plt.close(fig)
    tqdm.write(f'[{datetime.now():%H:%M:%S}] Figure and data: {figure_prefix}.{{pdf,png,csv.gz}}')

### 3.1 Reading the profiles

The y-axis shows mean MAF among retained variants in each occupied window. Window counts describe the number of retained variants, not the number of genomic bases. A line joins occupied windows across gaps; it does not imply variants in every intervening window.

### 3.1 读图

纵坐标表示每个有变异的窗口中，保留变异的平均 MAF。窗口计数描述保留变异的数量，而非基因组碱基数量。折线会跨越空缺连接有变异的窗口，不代表中间每个窗口都有变异。